In [172]:
# Una delle versioni migliori in giornata 12/05
# Feature extraction con CSP, sliding window e classificazione binaria (riposo vs attivazione). 
# Classificatore SVM con gridSearch.

from matplotlib import pyplot as plt
import mne
from mne.decoding import CSP
from mne_bids import BIDSPath, read_raw_bids
import numpy as np
from sklearn import preprocessing
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from scipy.signal import welch
from pathlib import Path
from util.preprocessing import create_sliding_windows
from util.preprocessing import create_window_labels

mne.set_log_level('WARNING') # Per rimuovere alcuni log superflui
eps = 1e-20
# Calcolo della del psd di una finestra tramite welch
# Ritorna: freqs=vettore monodimensionale con i valori discreti delle frequenze (da 0Hz fino a fs/2 generalmente)
# psd = potenza psd[i] associata a ciascuna frequenza freqs[i], in questo caso è bidimensionale e contiene la potenza associata a ciascun elettrodo
def compute_psd(window, fs):
    freqs, psd = welch(
        window, # Segnale 2d (numero elettrodi*numero campioni per elettrodo)
        fs=fs,  # Frequenza di campionamento
        nperseg=120,  # NumberXSegment: len dei segmenti usati da welch per stimare PSD. Alto=miglior stima, Basso=minor rumore
        axis=1
    )
    return freqs, psd

# Calcolo della potenza in una certa banda di frequenza (ottenuta tramite integrazione (il computer fa quindi la somma) del psd nel range desiderato)
# usage: bandpower(freqs, psd[x], (8, 13)), cioè calcola la potenza su un singolo canale
def bandpower(freqs, psd, band):
    fmin, fmax = band
    idx = np.logical_and(freqs >= fmin, freqs <= fmax)  # Quali indici prendere
    return np.trapezoid(psd[idx], freqs[idx])   # Approssima l'integrale tramite trapezio invece che rettangoli


# Calcola la potenza su più canali
def multibandpower(raw_data, freqs, psd, band, channels):
    idx = [raw_data.ch_names.index(ch) for ch in channels if ch in raw_data.ch_names]   #Serve raw per i channel names
    powers = []
    for i in idx:
        powers.append(bandpower(freqs, psd[i], band))
    return np.array(powers)

# Bande di interesse
bands = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta":  (13, 30),
}

root = "../data"
root = (Path(root).resolve())
runs = ["6", "10", "14"]   # Le run che vengono prese in considerazione
train_runs = ["6", "10"]
test_run = "14" 
first_person = 1
people = 10
all_accuracy = np.zeros(people)  # Per memorizzare l'accuratezza di ogni soggetto

# Eseguiamo la scansione di tutti i soggetti 
for i in range(first_person, first_person + people):
    subject = f"{i:03d}"
    X_train = []
    y_train = []

    X_test = []
    y_test = []

    # Per ogni soggetto eseguiamo la scansione sulle run di nostro interesse
    for run in runs:
        bids_path = BIDSPath(   # Specifichiamo il percorso del dataset e BIDS eseguirà correttamente la scansione 
            subject=subject,
            task="motion",
            run=run,
            datatype="eeg",
            root=root,
        )
        
        try:
            # Fase 1: lettura dei dati
            raw = read_raw_bids(bids_path, verbose=False)  
            events, event_id = mne.events_from_annotations(raw, verbose=False) 
            raw.load_data(verbose=False) # Carico i dati in memoria per poter filtrare ecc.
            raw.filter(l_freq=8, h_freq=30, verbose=False)  # Filtro passa banda 1-30 Hz
            raw.set_eeg_reference('average', projection=False, verbose=False)  # Riferimento medio
            event_map = {
                event_id['TASK4T0']: 1,
                event_id['TASK4T1']: 2,
                event_id['TASK4T2']: 3
            }


            # Fase 2: pre-processing
            window_size = 1.5  # Lunghezza finestra in secondi
            step_size = 0.3  # Lunghezza passo in secondi
            sfreq = raw.info['sfreq']  # frequenza di campionamento

            windows, window_samples, step_samples, total_samples = create_sliding_windows(
                raw,
                window_size,
                step_size
            )        

            C_channels = ["C1", "C2", "C3", "C4", "C5", "C6", "Cz"]
            CP_channels = [ch for ch in raw.ch_names if ch.startswith("Cp")]
            FC_channels = [ch for ch in raw.ch_names if ch.startswith("Fc")]
            T_channels =  [ch for ch in raw.ch_names if ch.startswith("T")]
            F_channels =  ["F1", "F2", "F3", "F4", "F5", "F6"]
            P_channels =  ["P1", "P2", "P3", "P4", "P5", "P6"]
            left_channel = ["C1", "C3", "C5", "Cz", "Fc1", "Fc3", "Fc5", "Fcz", "Cp1", "Cp3", "Cp5", "Cpz"]
            right_channel = ["C2", "C4", "C6", "Cz", "Fc2", "Fc4", "Fc6", "Fcz", "Cp2", "Cp4", "Cp6", "Cpz"]
            channels_of_interest = C_channels  + FC_channels + CP_channels 

            picks = mne.pick_channels(raw.ch_names, channels_of_interest)
            windows = windows[:, picks, :]

            y = create_window_labels(
                events,
                event_map,
                total_samples,
                window_samples,
                step_samples,
                threshold= 0.8
            )
            # Divido la classificazione in due step: prima distinguo tra stato di riposo e di attivazione.
            # In caso di attivazione, distinguo tra sinistra e destra. Per ora faccio solo la prima parte.
            y_rest_active = np.where(y == 1, 0, 1)      # Array rest/active
            mask_active = y != 1                        # Maschera per definire sinistra/destra
            y_active = y[mask_active]
            y_lr = np.where(y_active == 2, 0, 1)        # Array sinistra/destra (o sopra/sotto)
            # Assegno a y_binary le etichette desiderate (se voglio rest/active commento la seconda riga)
            y_binary = y_lr
            windows = windows[mask_active]

            # Memorizzo le varie potenze nelle bande di frequenza di interesse
            alpha_diff = []
            beta_diff = []
            features = []
            # Scorro su ogni finestra e memorizzo le feature ottenute tramite psd
            for i in range(len(y_binary)):
                freqs, psd = compute_psd(windows[i], sfreq)

                features = []

                for band_name, band in bands.items():

                    left_band = multibandpower(raw, freqs, psd, band, left_channel)
                    right_band = multibandpower(raw, freqs, psd, band, right_channel)
                    left_band = np.log10(left_band + eps)
                    right_band = np.log10(right_band + eps)
                    delta = left_band - right_band
                    features.append(delta)

                features = np.concatenate(features)  # (n_bands * n_channels,)

                if run in train_runs:
                    X_train.append(features)
                    y_train.append(y_binary[i])
                else:
                    X_test.append(features)
                    y_test.append(y_binary[i])

            # plt.figure()
            # print(alpha_diff)
            # plt.plot(np.array(alpha_diff) * 10, label="Alpha diff")
            # plt.plot(np.array(beta_diff) * 10, label="Beta diff")

            # plt.scatter(range(len(y_binary)), y_binary*4 - 2, label="Label (0/1)", marker="x")

            # plt.xlabel("Finestra")
            # plt.ylabel("Differenza potenza (L - R)")
            # plt.legend()
            # plt.show()

            # print(f"Soggetto {subject} run {run} - Campioni: {X_csp.shape[0]}, Feature per campione: {X_csp.shape[1]}, Etichette: {y.shape[0]}")

        except Exception as e:
            print(f"Errore {subject}: {e}")

    lda = LinearDiscriminantAnalysis()
    lda.fit(X_train, y_train)
    test_accuracy_lda = lda.score(X_test, y_test)
    print(f"Soggetto {subject} - Test accuracy LDA: {test_accuracy_lda:.4f}")

mid_accuracy = all_accuracy.mean()
print(f"Accuratezza media: {all_accuracy.mean()}")






Soggetto 001 - Test accuracy LDA: 0.6039
Soggetto 002 - Test accuracy LDA: 0.7468
Soggetto 003 - Test accuracy LDA: 0.5000
Soggetto 004 - Test accuracy LDA: 0.6558
Soggetto 005 - Test accuracy LDA: 0.5325
Soggetto 006 - Test accuracy LDA: 0.4870
Soggetto 007 - Test accuracy LDA: 0.7143
Soggetto 008 - Test accuracy LDA: 0.7013
Soggetto 009 - Test accuracy LDA: 0.4675
Soggetto 010 - Test accuracy LDA: 0.4091
Accuratezza media: 0.0
